# MIRI/LRS Slit Pipeline

In [1]:
import os
from collections import defaultdict
import boto3
from botocore.config import Config
import fsspec

# --------------------------Set Processing Steps--------------------------
# Individual pipeline stages can be turned on/off here.  Note that a later
# stage won't be able to run unless data products have already been
# produced from the prior stage.

# Pipeline processing
do_det1 = True  # calwebb_detector1
do_spec2 = True  # calwebb_spec2
do_spec3 = True  # calwebb_spec3
do_viz = True  # Visualize calwebb_spec3 results

# Background subtraction
bkg_sub = True  # Set as true to carry out nod subtraction for background removal (recommended)

# Optimal spectral extraction
opt_extract = False  # Set as true to carry out optimal spectral extraction in Stage 3 of the pipeline

# ------------------------Set CRDS context and paths----------------------
# Each version of the calibration pipeline is associated with a specific CRDS
# context file. The pipeline will select the appropriate context file behind
# the scenes while running. However, if you wish to override the default context
# file and run the pipeline with a different context, you can set that using
# the CRDS_CONTEXT environment variable. Here we show how this is done,
# although we leave the line commented out in order to use the default context.
# If you wish to specify a different context, uncomment the line below.
#os.environ['CRDS_CONTEXT'] = 'jwst_1322.pmap'  # CRDS context for 1.17.1

# Check whether the local CRDS cache directory has been set.
# If not, set it to the user home directory.
if (os.getenv('CRDS_PATH') is None):
    os.environ['CRDS_PATH'] = os.path.join(os.path.expanduser('~'), '.crds_cache')
    
# Check whether the CRDS server URL has been set.  If not, set it.
if (os.getenv('CRDS_SERVER_URL') is None):
    os.environ['CRDS_SERVER_URL'] = 'https://jwst-crds.stsci.edu'

# Print out CRDS path and context that will be used
print('CRDS local filepath:', os.environ['CRDS_PATH'])
print('CRDS file server:', os.environ['CRDS_SERVER_URL'])

# Use the entire available screen width for this notebook
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

# Basic system utilities for interacting with files
# ----------------------General Imports------------------------------------
import glob
import time
from pathlib import Path
from collections import defaultdict

# Numpy for doing calculations
import numpy as np

# -----------------------Astropy Imports-----------------------------------
# Astropy utilities for opening FITS and ASCII files and downloading demo files
from astropy.io import fits

# -----------------------Astroquery Imports-----------------------------------
from astroquery.mast import Observations

# -----------------------Plotting Imports----------------------------------
# Matplotlib for making plots
import matplotlib.pyplot as plt
from matplotlib import rc

# --------------JWST Calibration Pipeline Imports---------------------------
# Import the base JWST and calibration reference data packages
import jwst
import crds

# JWST pipelines (each encompassing many steps)
from jwst.pipeline import Detector1Pipeline
from jwst.pipeline import Spec2Pipeline
from jwst.pipeline import Spec3Pipeline

# JWST pipeline utilities
#from jwst import datamodels  # JWST datamodels
#from jwst.stpipe import Step  # Import the wrapper class for pipeline steps
from jwst.associations import asn_from_list as afl  # Tools for creating association files
from jwst.associations.lib.rules_level2_base import DMSLevel2bBase  # Definition of a Lvl2 association file
from jwst.associations.lib.rules_level3_base import DMS_Level3_Base  # Definition of a Lvl3 association file

# Print out pipeline version and CRDS context that will be used
print("JWST Calibration Pipeline Version = {}".format(jwst.__version__))
print("Using CRDS Context = {}".format(crds.get_context_name('jwst')))

CRDS local filepath: /home/dylan/.crds_cache
CRDS file server: https://jwst-crds.stsci.edu


/home/dylan/.miri_lrs_slit/lib/python3.13/site-packages/astropy/config/paths.py:55: AstropyUserWarning: XDG_CONFIG_HOME is set to '/home/dylan/.config', but the default location, /home/dylan/.astropy/config, already exists, and takes precedence. This environment variable will be ignored.
  return set_temp_config._get_dir_path(rootname)


JWST Calibration Pipeline Version = 2.0.0


CRDS - INFO -  Calibration SW Found: jwst 2.0.0 (/home/dylan/.miri_lrs_slit/lib/python3.13/site-packages/jwst-2.0.0.dist-info)


Using CRDS Context = jwst_1535.pmap


## Enable cloud access
This is helpful if you don't want to download everything

In [2]:
Observations.enable_cloud_dataset(provider='AWS')

# Uncomment line to disbale cloud access
# Observations.disable_cloud_dataset()

INFO: Using the S3 STScI public dataset [astroquery.mast.cloud]


## Define program detail for `astroquery`
Config below if you want to download files from MAST (or work from cloud files), otherwise skip and use local files

In [3]:
dataproduct_type = "spectrum"
proposal_id = "4583"
proposal_pi = "Scholz, Aleks"

obs_table = Observations.query_criteria(
    dataproduct_type=dataproduct_type,
    proposal_id=proposal_id,
    instrument_name="MIRI*",
    proposal_pi=proposal_pi,
)

unique_products = Observations.get_unique_product_list(obs_table)

filtered = Observations.filter_products(
    unique_products,
    calib_level="1",
    productType="SCIENCE",
    filters="P750L", # I think this is the filter used for the LRS mode
)

INFO: 39 of 464 products were duplicates. Only returning 425 unique product(s). [astroquery.mast.utils]
INFO: To return all products, use `Observations.get_product_list` [astroquery.mast.observations]


In [4]:
# creating some mappings to keep directories organized
targets = obs_table['target_name']
obsids = obs_table['obsid']
obs_ids = filtered['obs_id'] # this is confusing
parent_obsid = filtered['parent_obsid']

obsid_to_target = {}
for i in range(len(obsids)):
    obsid_to_target[obsids[i]] = targets[i]
obsid_to_target = {str(k): str(v) for k, v in obsid_to_target.items()}

obs_id_to_parent_obsid = {}
for i in range(len(obs_ids)):
    obs_id_to_parent_obsid[obs_ids[i]] = parent_obsid[i]
obs_id_to_parent_obsid = {str(k): str(v) for k, v in obs_id_to_parent_obsid.items()}

obs_id_to_target = {k: obsid_to_target[v] for k, v in obs_id_to_parent_obsid.items()}


In [5]:
# Group uncal files by object
s3_uris = Observations.get_cloud_uris(filtered)
grouped_uris = defaultdict(list)

for uri in s3_uris:
    group_key = uri.split('/')[-1].strip('_uncal.fits')
    grouped_uris[obs_id_to_target[group_key]].append(uri)
grouped_uris

defaultdict(list,
            {'CHA1110-7633': ['s3://stpubdata/jwst/public/jw04583/jw04583013001/jw04583013001_03103_00001_mirimage_uncal.fits',
              's3://stpubdata/jwst/public/jw04583/jw04583013001/jw04583013001_03103_00002_mirimage_uncal.fits'],
             'CHA1107-7626': ['s3://stpubdata/jwst/public/jw04583/jw04583014001/jw04583014001_03103_00002_mirimage_uncal.fits',
              's3://stpubdata/jwst/public/jw04583/jw04583014001/jw04583014001_03103_00001_mirimage_uncal.fits'],
             'CHA1110-7721': ['s3://stpubdata/jwst/public/jw04583/jw04583012001/jw04583012001_05101_00002_mirimage_uncal.fits',
              's3://stpubdata/jwst/public/jw04583/jw04583012001/jw04583012001_05101_00001_mirimage_uncal.fits'],
             'UHWJ247.95-24.78': ['s3://stpubdata/jwst/public/jw04583/jw04583007001/jw04583007001_05101_00002_mirimage_uncal.fits',
              's3://stpubdata/jwst/public/jw04583/jw04583007001/jw04583007001_05101_00001_mirimage_uncal.fits'],
             '

In [6]:
# Define output subdirectories to keep data products organized
base_dir = '/home/dylan/JWST/GO-4583/'
instrument = 'MIRI'

## `Detector1Pipeline`

In [7]:
# Boilerplate dictionary setup
det1dict = defaultdict(dict)

# Overrides for whether or not certain steps should be skipped (example)
#det1dict['emicorr']['skip'] = True

# Overrides for various reference files.
# If the files are not in the base local directory, provide full path.
#det1dict['dq_init']['override_mask'] = 'myfile.fits' # Bad pixel mask
#det1dict['saturation']['override_saturation'] = 'myfile.fits'  # Saturation
#det1dict['reset']['override_reset'] = 'myfile.fits'  # Reset
#det1dict['linearity']['override_linearity'] = 'myfile.fits'  # Linearity
#det1dict['rscd']['override_rscd'] = 'myfile.fits'  # RSCD
#det1dict['dark_current']['override_dark'] = 'myfile.fits'  # Dark current subtraction
#det1dict['jump']['override_gain'] = 'myfile.fits'  # Gain used by jump step
#det1dict['ramp_fit']['override_gain'] = 'myfile.fits'  # Gain used by ramp fitting step
#det1dict['jump']['override_readnoise'] = 'myfile.fits'  # Read noise used by jump step
#det1dict['ramp_fit']['override_readnoise'] = 'myfile.fits'  # Read noise used by ramp fitting step

# Turn on multi-core processing for jump step (off by default).  
# Choose what fraction of cores to use (quarter, half, or all)
det1dict['jump']['maximum_cores'] = 'quarter'

# Turn on detection of cosmic ray showers if desired (off by default)
det1dict['jump']['find_showers'] = True

# Adjust the flagging threshold for cosmic rays (default is 4.0)
det1dict['jump']['rejection_threshold'] = 4.

In [8]:
# stage 1 for all targets
# Loop over each target
for obs, s3_uris in grouped_uris.items():

    uncal_dir = os.path.join(base_dir, obs, instrument, '.uncal')  # temporary uncal files will go here
    det1_dir = os.path.join(base_dir, obs, instrument, 'stage1')   # calwebb_detector1 pipeline outputs will go here

    # Create desired output directories, if needed
    os.makedirs(uncal_dir, exist_ok=True)
    os.makedirs(det1_dir, exist_ok=True)

    print(f"\nStage 1 for {obs}...")
    
    # Download and process the files for this target object
    for uri in s3_uris:
        filename = os.path.basename(uri)
        uncal_path = os.path.join(uncal_dir, filename)

        stage1_wildcard = filename.replace("_uncal", "*")
        stage1_product_matches = glob.glob(os.path.join(det1_dir, stage1_wildcard))

        if stage1_product_matches:
            print(f"Found Stage 1 products for {filename}... Skipping...")
            continue 

        if os.path.exists(uncal_path):
            print(f"Found local uncal file {filename}...")
            
        # Following astroquery
        else:
            print(f"Downloading: {filename}")
            with fsspec.open(uri, "rb", anon=True, s3_additional_kwargs={'RequestPayer': 'requester'}) as cloud_file:
                with open(uncal_path, "wb") as local_file:
                    local_file.write(cloud_file.read())
        
        print(f"Running Detector1Pipeline on {filename}...")
        Detector1Pipeline.call(uncal_path, steps=det1dict, save_results=True, output_dir=det1_dir)

        # Clean raw file
        os.remove(uncal_path)

print("\nAll files successfully processed through Stage 1!")


Stage 1 for CHA1110-7633...
Found Stage 1 products for jw04583013001_03103_00001_mirimage_uncal.fits... Skipping...
Found Stage 1 products for jw04583013001_03103_00002_mirimage_uncal.fits... Skipping...

Stage 1 for CHA1107-7626...
Found Stage 1 products for jw04583014001_03103_00002_mirimage_uncal.fits... Skipping...
Found Stage 1 products for jw04583014001_03103_00001_mirimage_uncal.fits... Skipping...

Stage 1 for CHA1110-7721...
Found Stage 1 products for jw04583012001_05101_00002_mirimage_uncal.fits... Skipping...
Found Stage 1 products for jw04583012001_05101_00001_mirimage_uncal.fits... Skipping...

Stage 1 for UHWJ247.95-24.78...
Found Stage 1 products for jw04583007001_05101_00002_mirimage_uncal.fits... Skipping...
Found Stage 1 products for jw04583007001_05101_00001_mirimage_uncal.fits... Skipping...

Stage 1 for UGC0417+2832...
Found Stage 1 products for jw04583001001_06101_00002_mirimage_uncal.fits... Skipping...
Found Stage 1 products for jw04583001001_06101_00001_mirima

## `Spec2Pipeline`

In [9]:
# Set up a dictionary to define how the Spec2 pipeline should be configured

# Boilerplate dictionary setup
spec2dict = defaultdict(dict)

# Activate nod subtraction for background removal, if requested
if (bkg_sub is True):
    spec2dict['bkg_subtract']['skip'] = False
else:
    spec2dict['bkg_subtract']['skip'] = True
    
# Overrides for whether or not certain steps should be skipped (example)
#spec2dict['straylight']['skip'] = True

# Overrides for various reference files
# Files should be in the base local directory or provide full path
#spec2dict['assign_wcs']['override_distortion'] = 'myfile.asdf'  # Spatial distortion (ASDF file)
#spec2dict['assign_wcs']['override_specwcs'] = 'myfile.asdf'  # Spectral distortion (ASDF file)
#spec2dict['flat_field']['override_flat'] = 'myfile.fits'  # Pixel flatfield
#spec2dict['photom']['override_photom'] = 'myfile.fits'  # Photometric calibration array
#spec2dict['extract_1d']['override_extract1d'] = 'myfile.asdf'  # Spectral extraction parameters (ASDF file)
#spec2dict['extract_1d']['override_apcorr'] = 'myfile.asdf'  # Aperture correction parameters (ASDF file)

In [10]:
# Function to create association.
# scifile = list of science files
# bkgfile = list of background files
# asnfile = name of the association file
def writel2asn(scifile, bkgfile, asnfile):
    # Define the basic association of the science exposure
    asn = afl.asn_from_list([scifile], rule=DMSLevel2bBase)  # Wrap in array since input is single exposure

    # Add the background exposure
    asn['products'][0]['members'].append({'expname': bkgfile, 'exptype': 'background'})              

    # Write the association to a json file
    _, serialized = asn.dump()
    with open(asnfile, 'w') as outfile:
        outfile.write(serialized)

In [11]:
for obs in grouped_uris.keys():

    print(f"\nStage 2 for {obs}...")
    
    det1_dir = os.path.join(base_dir, obs, instrument, 'stage1')   # calwebb_detector1 pipeline outputs will go here
    spec2_dir = os.path.join(base_dir, obs, instrument, 'stage2')  # calwebb_spec2 pipeline outputs will go here

    # Create desired output directories, if needed
    os.makedirs(spec2_dir, exist_ok=True)
    
    # Get rate files from the Detector1 output folder
    rate_files = sorted(glob.glob(os.path.join(det1_dir, '*rate.fits')))
    
    # Use the absolute file paths
    for ii in range(len(rate_files)):
        rate_files[ii] = os.path.abspath(rate_files[ii])
    rate_files = np.array(rate_files)
    
    # Background files are the opposite nod position exposures
    bkg_files = rate_files[::-1]

    # Run the pipeline on the selected rate files one by one with the custom parameter dictionary
    if do_spec2:
        for ii, file in enumerate(rate_files):
            if bkg_sub:
                asnfile = os.path.join(det1_dir, Path(file).stem + '_asn.json')
                writel2asn(file, bkg_files[ii], asnfile)
                Spec2Pipeline.call(asnfile, steps=spec2dict, save_results=True, output_dir=spec2_dir)
            else:
                Spec2Pipeline.call(file, steps=spec2dict, save_results=True, output_dir=spec2_dir)
                
    else:
        print('Skipping Spec2 processing...')


Stage 2 for CHA1110-7633...


/home/dylan/.miri_lrs_slit/lib/python3.13/site-packages/jwst/associations/association.py:232: UserWarning: Input association file contains path information; note that this can complicate usage and/or sharing of such files.
  warnings.warn(err_str, UserWarning, stacklevel=1)
2026-05-16 17:34:31,897 - stpipe.step - INFO - PARS-RESAMPLESPECSTEP parameters found: /home/dylan/.crds_cache/references/jwst/miri/jwst_miri_pars-resamplespecstep_0001.asdf
2026-05-16 17:34:31,911 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-05-16 17:34:31,924 - stpipe.step - INFO - PARS-RESAMPLESPECSTEP parameters found: /home/dylan/.crds_cache/references/jwst/miri/jwst_miri_pars-resamplespecstep_0001.asdf
2026-05-16 17:34:31,936 - stpipe.pipeline - INFO - PARS-SPEC2PIPELINE parameters found: /home/dylan/.crds_cache/references/jwst/miri/jwst_miri_pars-spec2pipeline_0009.asdf
2026-05-16 17:34:31,967 - stpipe.step - INFO - Spe


Stage 2 for CHA1107-7626...


2026-05-16 17:34:44,451 - stpipe.step - INFO - Step Spec2Pipeline running with args ('/home/dylan/JWST/GO-4583/CHA1107-7626/MIRI/stage1/jw04583014001_03103_00001_mirimage_rate_asn.json',).
2026-05-16 17:34:44,498 - stpipe.step - INFO - Step Spec2Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/dylan/JWST/GO-4583/CHA1107-7626/MIRI/stage2
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  save_bsub: False
  fail_on_exception: True
  save_wfss_esec: False
  steps:
    assign_wcs:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: False
      output_use_index: True
      save_results: False
      skip: False
      suffix: None
      search_output_file: True
      input_dir: ''
      sip_approx: True
      sip_max_pix_error: 0.01
      sip_degree


Stage 2 for CHA1110-7721...


...

2026-05-16 17:35:30,642 - stpipe.step - INFO - WfssContamStep instance created.
2026-05-16 17:35:30,643 - stpipe.step - INFO - PhotomStep instance created.
2026-05-16 17:35:30,644 - stpipe.step - INFO - AdaptiveTraceModelStep instance created.
2026-05-16 17:35:30,645 - stpipe.step - INFO - PixelReplaceStep instance created.
2026-05-16 17:35:30,646 - stpipe.step - INFO - ResampleSpecStep instance created.
2026-05-16 17:35:30,647 - stpipe.step - INFO - CubeBuildStep instance created.
2026-05-16 17:35:30,650 - stpipe.step - INFO - Extract1dStep instance created.
2026-05-16 17:35:30,791 - stpipe.step - INFO - Step Spec2Pipeline running with args ('/home/dylan/JWST/GO-4583/UGC0422+2655/MIRI/stage1/jw04583002001_06101_00001_mirimage_rate_asn.json',).
2026-05-16 17:35:30,833 - stpipe.step - INFO - Step Spec2Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/dylan/JWST/GO-4583/UGC0422+2655/MIRI/stage2
  output_ext: .fits
  output_use_model: False



Stage 2 for UGC0433+2251...


2026-05-16 17:35:42,298 - stpipe.step - INFO - Step Spec2Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/dylan/JWST/GO-4583/UGC0433+2251/MIRI/stage2
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  save_bsub: False
  fail_on_exception: True
  save_wfss_esec: False
  steps:
    assign_wcs:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: False
      output_use_index: True
      save_results: False
      skip: False
      suffix: None
      search_output_file: True
      input_dir: ''
      sip_approx: True
      sip_max_pix_error: 0.01
      sip_degree: None
      sip_max_inv_pix_error: 0.01
      sip_inv_degree: None
      sip_npoints: 12
      slit_y_low: -0.55
      slit_y_high: 0.55
      nrs_ifu_slice_wcs: False
    badpix_selfcal:



Stage 2 for UGC0439+2642...


2026-05-16 17:35:53,796 - stpipe.step - INFO - Step Spec2Pipeline running with args ('/home/dylan/JWST/GO-4583/UGC0439+2642/MIRI/stage1/jw04583003001_06101_00001_mirimage_rate_asn.json',).
2026-05-16 17:35:53,841 - stpipe.step - INFO - Step Spec2Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/dylan/JWST/GO-4583/UGC0439+2642/MIRI/stage2
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  save_bsub: False
  fail_on_exception: True
  save_wfss_esec: False
  steps:
    assign_wcs:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: False
      output_use_index: True
      save_results: False
      skip: False
      suffix: None
      search_output_file: True
      input_dir: ''
      sip_approx: True
      sip_max_pix_error: 0.01
      sip_degree

## `Spec3Pipeline`

In [12]:
# Set up a dictionary to define how the Spec3 pipeline should be configured

# Boilerplate dictionary setup
spec3dict = defaultdict(dict)

# Overrides for whether or not certain steps should be skipped (example)
#spec3dict['outlier_detection']['skip'] = True

# Overrides for various reference files
# Files should be in the base local directory or provide full path
#spec3dict['extract_1d']['override_extract1d'] = 'myfile.json'  # Spectral extraction parameters (ASDF file)
#spec3dict['extract_1d']['override_apcorr'] = 'myfile.asdf'  # Aperture correction parameters (ASDF file)

The `pixel_replace` step is not run by default, but is recommended for mitigating the effect of bad pixels and other detector artifacts. The default method within the pipeline is to fit a local profile to adjacent pixels and interpolate the flux within the problem region.

In [13]:
# Run pixel replacement code to interpolate values for otherwise bad pixels.
spec3dict['pixel_replace']['skip'] = False
#spec3dict['pixel_replace']['save_results'] = True   # Enable to write out these files for spot checking

As of Version 1.18.0, the JWST calibration pipeline includes the capability of carrying out optimal spectral extraction using an empirical PSF in the JWST CRDS. To use this functionality, a number of parameter settings must be provided to the pipeline. The advantage of this approach is that, in principle, outliers and bad pixels have little to no effect on the extracted fluxes; as such, the `resample_spec` and `pixel_replace` steps can be skipped. Moreover, in cases where the signal-to-noise ratio of the data is poor, the use of a predefined PSF model greatly enhances the quality of the measured spectrum.

For targets with relatively bright continua, the `optimize_psf_location` parameter should be set to True to allow the fitting algorithm to determine the proper centroid position of the spectral trace. However, in the case of faint targets, the parameter should instead be set to False, as there may not be sufficient signal to reliably center the PSF model. 

The empirical PSF model in the JWST CRDS is periodically updated using additional observations to improve the signal-to-noise ratio and overall quality of the PSF model across the LRS slit wavelength range.

In [14]:
opt_extract = False
# Use optimal spectral extraction, trying this since objects are faint point sources. you know just play around have fun see what works. okay i have played around it is all the same
if opt_extract:
    spec3dict['resample_spec']['skip'] = True
    spec3dict['pixel_replace']['skip'] = True
    spec3dict['extract_1d'] = {'extraction_type': 'optimal', 'use_source_posn': True, 'model_nod_pair': True, 'optimize_psf_location': False, 'save_profile': True, 'save_residual_image': True}

In [15]:
def writel3asn(cal_files, asnfile):
    # Define the basic association of science files
    asn = afl.asn_from_list(cal_files, rule=DMS_Level3_Base, product_name='Stage3')

    # Write the association to a json file
    _, serialized = asn.dump()
    with open(asnfile, 'w') as outfile:
        outfile.write(serialized)

In [16]:
for obs in grouped_uris.keys():

    print(f"\nStage 3 for {obs}...")
    
    spec2_dir = os.path.join(base_dir, obs, instrument, 'stage2')   # calwebb_spec2 pipeline outputs will go here
    spec3_dir = os.path.join(base_dir, obs, instrument, 'stage3')  # calwebb_spec3 pipeline outputs will go here

    # Create desired output directories, if needed
    os.makedirs(spec3_dir, exist_ok=True)
    
    # Get cal files from the Spec2 output folder
    cal_files = sorted(glob.glob(os.path.join(spec2_dir, '*cal.fits')))
    
    # Use the absolute file paths
    for ii in range(len(cal_files)):
        cal_files[ii] = os.path.abspath(cal_files[ii])
    cal_files = np.array(cal_files)
    
    # Create association file
    asnfile = os.path.join(spec3_dir, 'stage3_asn.json')
    writel3asn(cal_files, asnfile)
    
    if do_spec3:
        Spec3Pipeline.call(asnfile, steps=spec3dict, save_results=True, output_dir=spec3_dir)
    else:
        print('Skipping Spec3 processing...')

/home/dylan/.miri_lrs_slit/lib/python3.13/site-packages/jwst/associations/association.py:232: UserWarning: Input association file contains path information; note that this can complicate usage and/or sharing of such files.
  warnings.warn(err_str, UserWarning, stacklevel=1)
2026-05-16 17:36:05,194 - stpipe.step - INFO - PARS-RESAMPLESPECSTEP parameters found: /home/dylan/.crds_cache/references/jwst/miri/jwst_miri_pars-resamplespecstep_0001.asdf
2026-05-16 17:36:05,210 - stpipe.pipeline - INFO - PARS-SPEC3PIPELINE parameters found: /home/dylan/.crds_cache/references/jwst/miri/jwst_miri_pars-spec3pipeline_0007.asdf
2026-05-16 17:36:05,229 - stpipe.step - INFO - Spec3Pipeline instance created.
2026-05-16 17:36:05,230 - stpipe.step - INFO - AssignMTWcsStep instance created.
2026-05-16 17:36:05,231 - stpipe.step - INFO - MasterBackgroundStep instance created.
2026-05-16 17:36:05,232 - stpipe.step - INFO - OutlierDetectionStep instance created.
2026-05-16 17:36:05,233 - stpipe.step - INFO - 


Stage 3 for CHA1110-7633...


2026-05-16 17:36:05,378 - stpipe.step - INFO - Step Spec3Pipeline running with args ('/home/dylan/JWST/GO-4583/CHA1110-7633/MIRI/stage3/stage3_asn.json',).
2026-05-16 17:36:05,399 - stpipe.step - INFO - Step Spec3Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/dylan/JWST/GO-4583/CHA1110-7633/MIRI/stage3
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  steps:
    assign_mtwcs:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: True
      output_use_index: True
      save_results: False
      skip: False
      suffix: assign_mtwcs
      search_output_file: True
      input_dir: ''
    master_background:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: Tru


Stage 3 for CHA1107-7626...


2026-05-16 17:36:11,644 - stpipe.step - INFO - Step Spec3Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/dylan/JWST/GO-4583/CHA1107-7626/MIRI/stage3
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  steps:
    assign_mtwcs:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: True
      output_use_index: True
      save_results: False
      skip: False
      suffix: assign_mtwcs
      search_output_file: True
      input_dir: ''
    master_background:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: True
      output_use_index: True
      save_results: False
      skip: True
      suffix: None
      search_output_file: True
      input_dir: ''
      median

...

2026-05-16 17:36:35,696 - stpipe.step - INFO - Step Spec3Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/dylan/JWST/GO-4583/UGC0422+2655/MIRI/stage3
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  steps:
    assign_mtwcs:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: True
      output_use_index: True
      save_results: False
      skip: False
      suffix: assign_mtwcs
      search_output_file: True
      input_dir: ''
    master_background:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: True
      output_use_index: True
      save_results: False
      skip: True
      suffix: None
      search_output_file: True
      input_dir: ''
      median


Stage 3 for UGC0433+2251...


2026-05-16 17:36:42,117 - stpipe.step - INFO - Step Spec3Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/dylan/JWST/GO-4583/UGC0433+2251/MIRI/stage3
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  steps:
    assign_mtwcs:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: True
      output_use_index: True
      save_results: False
      skip: False
      suffix: assign_mtwcs
      search_output_file: True
      input_dir: ''
    master_background:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: True
      output_use_index: True
      save_results: False
      skip: True
      suffix: None
      search_output_file: True
      input_dir: ''
      median


Stage 3 for UGC0439+2642...


2026-05-16 17:36:48,568 - stpipe.step - INFO - Step Spec3Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/dylan/JWST/GO-4583/UGC0439+2642/MIRI/stage3
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  steps:
    assign_mtwcs:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: True
      output_use_index: True
      save_results: False
      skip: False
      suffix: assign_mtwcs
      search_output_file: True
      input_dir: ''
    master_background:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: True
      output_use_index: True
      save_results: False
      skip: True
      suffix: None
      search_output_file: True
      input_dir: ''
      median